# CVRP Project — Test de Hill Climbing

Ce notebook charge une instance CVRP et compare Clarke–Wright, utilisé comme
solution de départ, au résultat obtenu après l'amélioration par Hill Climbing.
Exécutez les cellules dans l'ordre.

In [11]:
from pathlib import Path
from time import perf_counter

from algorithms.hill_climbing import _initial_solution, hill_climbing
from common import distance_matrix, parse_vrp, solution_cost, validate_solution

# Le chemin est relatif à la racine du projet. Modifiez-le pour tester une autre instance.
instance_path = Path("data/A-n80-k10.vrp")
if not instance_path.exists():
    raise FileNotFoundError(f"Instance introuvable : {instance_path.resolve()}")

instance = parse_vrp(instance_path)
print(f"Instance : {instance.get('name', instance_path.stem)}")
print(f"Clients : {len(instance['demands']) - 1} | Véhicules : {instance['vehicles']} | Capacité : {instance['capacity']}")

Instance : A-n80-k10
Clients : 79 | Véhicules : 10 | Capacité : 100


## Solution initiale Clarke–Wright

Clarke–Wright regroupe les clients en tournées faisables à partir des économies de distance. Hill Climbing part de ces tournées, puis cherche des modifications locales qui réduisent le coût. Cette cellule montre le point de départ de l'amélioration.

In [12]:
# Construire et afficher exactement la solution de départ utilisée par Hill Climbing
distances = distance_matrix(instance)
initial_routes = _initial_solution(instance, distances)
initial_cost = solution_cost(initial_routes, instance['depot'], distances)

print(f"Coût Clarke–Wright : {initial_cost}")
print(f"Solution faisable : {validate_solution(initial_routes, instance)}")
for number, route in enumerate(initial_routes, start=1):
    if route:
        print(f"Véhicule {number} : dépôt -> {' -> '.join(map(str, route))} -> dépôt ({len(route)} clients)")

Coût Clarke–Wright : 1831
Solution faisable : True
Véhicule 1 : dépôt -> 22 -> 8 -> 11 -> 64 -> 2 -> dépôt (5 clients)
Véhicule 2 : dépôt -> 63 -> 24 -> 6 -> 28 -> 60 -> 32 -> 18 -> 65 -> dépôt (8 clients)
Véhicule 3 : dépôt -> 25 -> 7 -> 31 -> 79 -> 9 -> 38 -> 3 -> 35 -> 72 -> dépôt (9 clients)
Véhicule 4 : dépôt -> 12 -> 53 -> 29 -> 80 -> 19 -> 49 -> 15 -> dépôt (7 clients)
Véhicule 5 : dépôt -> 13 -> 45 -> 30 -> 61 -> 40 -> 75 -> 14 -> dépôt (7 clients)
Véhicule 6 : dépôt -> 73 -> 55 -> 10 -> 56 -> 16 -> 42 -> 26 -> 76 -> 21 -> 47 -> dépôt (10 clients)
Véhicule 7 : dépôt -> 34 -> 48 -> 57 -> 70 -> 66 -> 36 -> 27 -> 20 -> 58 -> 62 -> 17 -> 44 -> 69 -> dépôt (13 clients)
Véhicule 8 : dépôt -> 52 -> 78 -> 4 -> 54 -> 67 -> 68 -> 39 -> 37 -> dépôt (8 clients)
Véhicule 9 : dépôt -> 41 -> 43 -> 74 -> 50 -> dépôt (4 clients)
Véhicule 10 : dépôt -> 59 -> 33 -> 5 -> 23 -> 46 -> 51 -> 77 -> 71 -> dépôt (8 clients)


In [10]:
# Exécuter l'algorithme et mesurer son temps de calcul
start = perf_counter()
result = hill_climbing(instance, max_iterations=10_000)
elapsed = perf_counter() - start

print(f"Coût total : {result['cost']}")
print(f"Itérations : {result['iterations']}")
print(f"Temps : {elapsed:.2f} s")
print(f"Solution faisable : {validate_solution(result['routes'], instance)}")

Coût total : 1818
Itérations : 6
Temps : 0.38 s
Solution faisable : True


In [7]:
# Détail des tournées (les identifiants sont ceux de l'instance)
for number, route in enumerate(result['routes'], start=1):
    if route:  # Ignore les éventuelles tournées inutilisées
        print(f"Véhicule {number} : dépôt -> {' -> '.join(map(str, route))} -> dépôt ({len(route)} clients)")

Véhicule 1 : dépôt -> 22 -> 8 -> 11 -> 64 -> 2 -> dépôt (5 clients)
Véhicule 2 : dépôt -> 63 -> 24 -> 6 -> 60 -> 28 -> 18 -> 32 -> 65 -> dépôt (8 clients)
Véhicule 3 : dépôt -> 25 -> 7 -> 31 -> 79 -> 9 -> 38 -> 3 -> 35 -> 72 -> dépôt (9 clients)
Véhicule 4 : dépôt -> 12 -> 53 -> 29 -> 80 -> 19 -> 49 -> 15 -> dépôt (7 clients)
Véhicule 5 : dépôt -> 75 -> 40 -> 61 -> 30 -> 45 -> 13 -> 41 -> dépôt (7 clients)
Véhicule 6 : dépôt -> 73 -> 55 -> 10 -> 56 -> 16 -> 42 -> 26 -> 76 -> 21 -> 47 -> dépôt (10 clients)
Véhicule 7 : dépôt -> 34 -> 48 -> 57 -> 70 -> 66 -> 36 -> 27 -> 20 -> 58 -> 62 -> 17 -> 44 -> 69 -> dépôt (13 clients)
Véhicule 8 : dépôt -> 52 -> 4 -> 78 -> 54 -> 67 -> 68 -> 39 -> 37 -> dépôt (8 clients)
Véhicule 9 : dépôt -> 14 -> 43 -> 74 -> 50 -> dépôt (4 clients)
Véhicule 10 : dépôt -> 59 -> 33 -> 5 -> 23 -> 46 -> 51 -> 77 -> 71 -> dépôt (8 clients)


## Comparaison avec la solution de référence

Le fichier `.sol` fourni avec l'instance contient une solution de référence de coût 1763. La comparaison ci-dessous calcule aussi l'écart relatif de Hill Climbing par rapport à cette valeur.

In [8]:
import re

solution_path = instance_path.with_suffix('.sol')
if not solution_path.exists():
    raise FileNotFoundError(f"Solution de référence introuvable : {solution_path.resolve()}")

solution_text = solution_path.read_text(encoding='utf-8')
match = re.search(r'^Cost\s+(\d+)\s*$', solution_text, flags=re.MULTILINE | re.IGNORECASE)
if match is None:
    raise ValueError(f"Aucun coût de référence trouvé dans {solution_path}")
reference_cost = int(match.group(1))
gap_percent = (result['cost'] - reference_cost) / reference_cost * 100

print(f"Coût de référence : {reference_cost}")
print(f"Coût Hill Climbing : {result['cost']}")
print(f"Écart : {result['cost'] - reference_cost:+d} ({gap_percent:+.2f} %)")
if result['cost'] < reference_cost:
    print("Hill Climbing a trouvé une solution moins coûteuse que la référence fournie.")
elif result['cost'] == reference_cost:
    print("Hill Climbing égale le coût de la référence fournie.")
else:
    print("Hill Climbing est au-dessus du coût de la référence fournie.")

Coût de référence : 1763
Coût Hill Climbing : 1818
Écart : +55 (+3.12 %)
Hill Climbing est au-dessus du coût de la référence fournie.
